[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/06_%E5%8F%97%E6%B3%A8%E7%8E%87%E3%81%AE%E5%8C%BA%E9%96%93%E6%8E%A8%E5%AE%9A%E3%81%A8%E6%A4%9C%E5%AE%9A.ipynb)

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに data/ が無ければ公開リポジトリ(raw)から読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
print('準備OK')

# 実践マーケ-06. 受注率の区間推定と検定（母比率）

**舞台設定**：先期の商談400件のうち成約は何件か。「受注率◯%」と一点で言うだけでなく、**どれくらいの幅で信用できるか**、そして **目標(例:20%)を本当に超えたと言えるか** を判断します。

**使う統計（読む=3級／本質=2級）**：母比率の信頼区間・母比率の仮説検定。

### 📋 使うデータ：`sales_btob.csv`（架空のBtoB商談400件）
1行＝商談1件。`受注` は 1=成約 / 0=失注。この列の平均が受注率です。

| 商談ID | 獲得チャネル | 商談金額 | 受注 |
|---|---|---|---|
| D0001 | 紹介 | 1,211,000 | 0 |

## 🎯 この章でできるようになること
- 受注率を「点推定」だけでなく **95%信頼区間** で表せる
- 「目標を超えた」と言えるかを **母比率の検定** で確かめられる
- 標本サイズ n が増えると区間が狭くなる仕組みを理解する

> **前提**：統計3級（割合・正規分布）→ 本質は2級（区間推定・検定）　/　**所要**：約25分　/　**レベル**：実践（読む3級・本質2級）

## 1. 受注率の点推定

In [ ]:
btob = load('sales_btob.csv')
n = len(btob); x = int(btob['受注'].sum()); p = x / n
print(f'商談 {n} 件中 成約 {x} 件  → 受注率（点推定） {p:.1%}')

## 2. 受注率の95%信頼区間（母比率の区間推定）

同じ営業活動をくり返しても受注率は毎回ぶれます。母比率の95%信頼区間は `p̂ ± 1.96 × √(p̂(1−p̂)/n)`。「本当の受注率」が入る範囲の目安です。

In [ ]:
se = np.sqrt(p*(1-p)/n)                 # 標準誤差
lo, hi = p - 1.96*se, p + 1.96*se
print(f'標準誤差 SE = {se:.4f}')
print(f'95%信頼区間: {lo:.1%} 〜 {hi:.1%}（半幅 ±{1.96*se:.1%}）')

## 3. 「目標20%を超えた」と言える？（母比率の検定）

- H₀（帰無仮説）：本当の受注率は20%（= 目標ちょうど）
- H₁（対立仮説）：20%と違う

H₀のもとでの標準誤差を使って z 統計量と p 値を出します。

In [ ]:
from scipy import stats
p0 = 0.20
se0 = np.sqrt(p0*(1-p0)/n)              # H0のもとでの標準誤差
z = (p - p0) / se0
pval = 2 * (1 - stats.norm.cdf(abs(z)))  # 両側
print(f'z = {z:.3f},  p値 = {pval:.4f}')
print('→ 有意水準5%で', ('有意差あり（20%と異なる）' if pval < 0.05 else '有意差なし（20%と言い切れない）'))

## 4. 標本サイズ n と区間の幅

n を増やすほど信頼区間は狭く（＝推定が精密に）なります。半幅は √n に反比例します。

In [ ]:
for nn in [100, 400, 1600]:
    half = 1.96 * np.sqrt(p*(1-p)/nn)
    print(f'n={nn:>5}: 半幅 ±{half:.1%}')
print('→ n を4倍にすると半幅は約1/2（√4分の1）')

## 🧭 まとめ
- 率は **点推定＋信頼区間** のセットで報告する（「20%（±◯%）」のように）。
- 「目標を超えたか」は **母比率の検定**（z・p値）で判断する。
- 精度（区間の幅）は **n** で決まる。n を増やすほど狭くなる。

> ⚠️ **よくある間違い**
> - 95%信頼区間は「この1本が95%の確率で真値を含む」ではなく「同じ手順をくり返すと95%が真値を含む」。
> - **有意 ≠ 重要**。n が大きいとごくわずかな差でも有意になる。実務的な大きさ（効果量）も見る。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: 商談200件中50件成約のときの受注率を ans に
ans = None   # 例: 50/200
_check('Q1 受注率', ans, 50/200)

In [ ]:
import numpy as np
# Q2: p=0.25, n=200 のときの95%信頼区間の半幅 1.96×√(p(1−p)/n) を ans に
p, n = 0.25, 200
ans = None   # 例: 1.96*np.sqrt(p*(1-p)/n)
_check('Q2 区間の半幅', ans, 1.96*np.sqrt(p*(1-p)/n))

In [ ]:
import numpy as np
# Q3: 受注率0.25・目標0.20・n=200。H0(p=0.20)のもとでの z=(0.25−0.20)/√(0.20·0.80/200) を ans に
ans = None   # 例: (0.25-0.20)/np.sqrt(0.20*0.80/200)
_check('Q3 z統計量', ans, (0.25-0.20)/np.sqrt(0.20*0.80/200))

---
## 🏆 ワークシート課題

**課題1.** `獲得チャネル` が「紹介」の商談だけにしぼって、受注率と95%信頼区間を出そう。全体と比べて高い？

**課題2.** 目標を25%にしたら、検定の結論は変わる？ `p0` を変えて確かめよう。

**課題3.**（考察）信頼区間の半幅を「±3%以内」にしたい。必要な n はおよそ何件か、半幅の式 `1.96√(p(1−p)/n)` から逆算してみよう。

> 🔑 **解答例は別ページ**（ネタバレ防止）👉 **[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/06_%E5%8F%97%E6%B3%A8%E7%8E%87%E3%81%AE%E5%8C%BA%E9%96%93%E6%8E%A8%E5%AE%9A%E3%81%A8%E6%A4%9C%E5%AE%9A.md)**
>
> 自分で解いてから開きましょう。

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 点推定 | 1つの値での推定（受注率＝25% など） |
| 信頼区間 | 真値が入る範囲の目安（95%CIなど） |
| 標準誤差 SE | 推定値のばらつき。√(p(1−p)/n) |
| 帰無仮説 H₀ | 「差がない」とする仮定。検定で棄却を狙う |
| p値 | H₀が正しいとき今回以上の差が出る確率 |

```python
se = np.sqrt(p*(1-p)/n); ci = (p-1.96*se, p+1.96*se)   # 母比率の95%CI
z = (p-p0)/np.sqrt(p0*(1-p0)/n)                          # 母比率の検定
```